In [75]:
import os
import csv
import json
import oracledb
from langchain_core.documents import Document
from langchain_oracledb.vectorstores import oraclevs
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_oracledb.vectorstores.oraclevs import OracleVS
from langchain_core.vectorstores.base import VectorStoreRetriever
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_oracledb.retrievers.text_search import create_text_index, OracleTextSearchRetriever

In [76]:
username = "system"
password = "oracle"
dsn = "192.168.1.248:1521/FREEPDB1"

In [77]:
try:
    connection = oracledb.connect(user=username, password=password, dsn=dsn)
    print("Connection successful!")

except oracledb.Error as e:
    error_obj, = e.args
    print(f"Oracle Error: {error_obj.message}")

except Exception as e:
    print(f"Undefined Error: {e}")

Connection successful!


In [78]:
corpus_path = os.path.join(os.path.dirname(os.getcwd()), "scifact", "corpus.jsonl")
corpus_path

'd:\\IE103_Final_Project\\scifact\\corpus.jsonl'

In [79]:
documents_langchain = []

with open(corpus_path, "r", encoding="utf-8") as document_jsonl_list:
    for line in document_jsonl_list:
        doc = json.loads(line)
        metadata = {"id": doc["_id"], "title": doc["title"]}
        doc_langchain = Document(page_content=doc["text"], metadata=metadata)
        documents_langchain.append(doc_langchain)

In [80]:
model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3600.86it/s]


In [81]:
vector_store = OracleVS(
    client=connection,
    embedding_function=model,
    table_name="SciFact",
    distance_strategy=DistanceStrategy.COSINE,
)

In [82]:
oraclevs.create_index(
    client=connection,
    vector_store=vector_store,
    params={"idx_name": "hnsw_scifact", "idx_type": "HNSW"},
)

In [83]:
def SemanticRetrieveTopK(vector_store: OracleVS, k: int) -> VectorStoreRetriever:
    retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k}
    )
    return retriever

In [84]:
create_text_index(
    client=connection,
    idx_name="keyword_scifact",
    vector_store=vector_store,
)

In [85]:
def KeywordRetrieveTopK(vector_store: OracleVS, k: int) -> OracleTextSearchRetriever:
    retriever = OracleTextSearchRetriever(
        vector_store=vector_store,
        k=k,
        fuzzy=True
    )
    return retriever

In [86]:
def ChunksToDocuments(retrieval_results: list[Document], documents_langchain: list[Document]) -> list[Document]:
    corpus_results = set()
    for result in retrieval_results:
        corpus_id = result.metadata["corpus_id"]
        corpus_results.add(corpus_id)

    document_results = []
    for document in documents_langchain:
        corpus_id = document.metadata["id"]
        if corpus_id in corpus_results:
            document_results.append(document)

    return document_results

In [87]:
def DocumentsToCorpusID(document_results: list[Document]) -> list[str]:
    corpus_ids = []
    for document in document_results:
        corpus_id = document.metadata["id"]
        corpus_ids.append(corpus_id)
    
    return corpus_ids

In [88]:
test_path = os.path.join(os.path.dirname(os.getcwd()), "scifact", "qrels", "test.tsv")
test_path

'd:\\IE103_Final_Project\\scifact\\qrels\\test.tsv'

In [89]:
test = {}

with open(test_path, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f, delimiter='\t')

    for row in reader:
        query_id = row["query-id"]
        corpus_id = row["corpus-id"]
        test[query_id] = corpus_id

In [90]:
queries_path = os.path.join(os.path.dirname(os.getcwd()), "scifact", "queries.jsonl")
queries_path

'd:\\IE103_Final_Project\\scifact\\queries.jsonl'

In [91]:
queries = {}

with open(queries_path, "r", encoding="utf-8") as document_jsonl_list:
    for line in document_jsonl_list:
        doc = json.loads(line)
        query_id = doc["_id"]
        text = doc["text"]
        queries[query_id] = text

In [92]:
for k in range(5, 51, 5):
    correct = 0
    semantic_retriever = SemanticRetrieveTopK(vector_store, k)
    keyword_retriever = KeywordRetrieveTopK(vector_store, k)

    for i, (query_id, corpus_id) in enumerate(test.items()):
        semantic_retrieval_results = semantic_retriever.invoke(queries[query_id])
        semantic_document_results = ChunksToDocuments(semantic_retrieval_results, documents_langchain)
        semantic_corpus_ids = DocumentsToCorpusID(semantic_document_results)

        keyword_retrieval_results = keyword_retriever.invoke(queries[query_id])
        keyword_document_results = ChunksToDocuments(keyword_retrieval_results, documents_langchain)
        keyword_corpus_ids = DocumentsToCorpusID(keyword_document_results)

        corpus_ids = set(semantic_corpus_ids + keyword_corpus_ids)

        if corpus_id in corpus_ids:
            correct += 1

    print(f"Hit@{k}: {correct / 300 * 100:.2f}%")

Hit@5: 78.67%
Hit@10: 82.33%
Hit@15: 84.67%
Hit@20: 86.33%
Hit@25: 87.67%
Hit@30: 88.00%
Hit@35: 89.00%
Hit@40: 89.67%
Hit@45: 91.33%
Hit@50: 92.00%
